# Задание 1
Ваша задача — получить эмбеддинги картинки и текста, относящиеся к одной и той же сущности. 
Скачайте датасет с исходными данными. 

У вас есть таблица `data/items.csv` с номером товара `id`, текстом `text`, названием картинки `image_path` и лейблом `label` в одной строке. Все картинки хранятся в отдельной директории `data/images/`, а названия файлов соответствуют указанным в таблице.
Напишите класс-загрузчик на базе `Dataset`, выполняющий токенизацию и загрузку картинок с подготовкой (масштабирование, нормализация, перевод в тензор) для выбранной модели. На вход он принимает путь до таблицы с данными `data/items.csv`, названия текстовой и картиночной моделей. Возвращает токенизированный текст и картинки в виде словаря с ключами `image, input_ids, attention_mask, label`.
* Создайте набор базовых трансформаций для изображений из `Resize, ToTensor, Normalize`.
* Используйте конфигурацию модели из `timm` для подготовки изображений и задания аргументов аугментаций.

In [16]:
import numpy as np
import pandas as pd
from functools import partial
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

import albumentations as A
from albumentations.pytorch import ToTensorV2  
from transformers import AutoTokenizer

In [17]:
class MultimodalDataset(Dataset):
    def __init__(self, df, text_model, image_model):
        self.df = df
        self.image_cfg = timm.get_pretrained_cfg(image_model)
        self.tokenizer = AutoTokenizer.from_pretrained(text_model)
        self.transform = T.Compose([
            T.Resize(self.image_cfg.input_size[1:]),
            T.ToTensor(),
            T.Normalize(mean=self.image_cfg.mean,
                        std=self.image_cfg.std)
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.loc[idx, "text"]
        label = self.df.loc[idx, "label"]
        tokenized_input = self.tokenizer(text,
                                         padding="max_length",
                                         truncation=True)

        img_path = self.df.loc[idx, "image_path"]
        image = Image.open(f"../data/images/{img_path}").convert('RGB')
        image = self.transform(image)
  
        return {
            "label": label,
            "image": image,
            "input_ids": torch.tensor(tokenized_input["input_ids"]),
            "attention_mask": torch.tensor(tokenized_input["attention_mask"])
        } 

# Задание 2
Допустим, вы выбрали текстовую и картиночную модели, сохранив их в переменные `text_model` и `image_model`. Теперь реализуйте аугментации для картинок с помощью `albumentations`:
* Используйте по одной аугментации из афинных, dropout и цветовых преобразований в дополнение к масштабированию и переводу в тензор.
* Укажите произвольные параметры аугментаций.
* Напишите реализацию `collate_fn`, которая будет использована внутри загрузчика данных. На вход она должна принимать аугментации и токенизатор для модели, возвращать словарь с ключами `image, input_ids, attention_mask, label`.
* Используйте конфиг картиночной модели для задания параметров аугментаций.
* Токенизируйте тексты в `collate_fn`.
* Передайте `collate_fn` с помощью модуля `partial` в загрузчик данных.
* `collate_fn` должна возвращать тензор.

In [18]:
class MultimodalDataset(Dataset):
    def __init__(self, df, text_model, image_model, transforms):
        self.df = df
        self.image_cfg = timm.get_pretrained_cfg(image_model)
        self.tokenizer = AutoTokenizer.from_pretrained(text_model)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df.loc[idx, "text"]
        label = self.df.loc[idx, "label"]

        img_path = self.df.loc[idx, "image_path"]
        image = Image.open(f"../data/images/{img_path}").convert('RGB')
        image = self.transforms(image=np.array(image))["image"]

        return {"label": label, "image": image, "text": text}

In [19]:
def collate_fn(batch, tokenizer):
    texts = [item["text"] for item in batch]
    images = torch.stack([item["image"] for item in batch])
    #print('\nDEBUG:', [item["label"] for item in batch],'\n')
    labels = torch.LongTensor([int(item["label"].split()[1]) for item in batch])

    tokenized_input = tokenizer(texts,
                                return_tensors="pt",
                                padding="max_length",
                                truncation=True)
    return {
        "label": labels,
        "image": images,
        "input_ids": tokenized_input["input_ids"],
        "attention_mask": tokenized_input["attention_mask"]
    }

In [20]:
text_model = "bert-base-uncased"
image_model = 'tf_efficientnet_b0'
tokenizer = AutoTokenizer.from_pretrained(text_model)
cfg = timm.get_pretrained_cfg(image_model)

transforms = A.Compose(
    [
        A.SmallestMaxSize(max_size=max(cfg.input_size[1], cfg.input_size[2]),
                          p=1.0),
        A.RandomCrop(height=cfg.input_size[1], width=cfg.input_size[2], p=1.0),
        A.Affine(scale=(0.8, 1.2),
                 rotate=(-15, 15),
                 translate_percent=(-0.1, 0.1),
                 shear=(-10, 10),
                 fill=0,
                 p=0.8),
        A.CoarseDropout(num_holes_range=(2, 8),
                        hole_height_range=(int(0.07 * cfg.input_size[1]),
                                           int(0.15 * cfg.input_size[1])),
                        hole_width_range=(int(0.1 * cfg.input_size[2]),
                                          int(0.15 * cfg.input_size[2])),
                        fill=0,
                        p=0.5),
        A.ColorJitter(
            brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.7),
        A.Normalize(mean=cfg.mean, std=cfg.std),
        A.ToTensorV2(p=1.0)  # конвертируем numpy HxWxC в torch.Tensor CxHxW
    ]
)

df = pd.read_csv("../data/items.csv")
ds = MultimodalDataset(df,
                       text_model=text_model,
                       image_model=image_model,
                       transforms=transforms)

loader = DataLoader(ds,
                    batch_size=1,
                    shuffle=False,
                    collate_fn=partial(collate_fn, tokenizer=tokenizer)) 

df

,text,image_path,label
0,3 лица,3_faces.png,label 1
1,буквальный хот-дог,corgi.jpg,label 2


In [21]:
for el in loader:
    print(el.keys())

dict_keys(['label', 'image', 'input_ids', 'attention_mask'])
dict_keys(['label', 'image', 'input_ids', 'attention_mask'])


Это самый простой вариант того, как может выглядеть сбор и загрузка данных. В зависимости от того, где расположены данные и каков их объём, некоторые части кода можно адаптировать соответствующим образом. Например, для валидационного датасета из аугментаций нужно использовать только `SmallestMaxSize, CenterCrop, Normalize`.

А ещё в таких мультимодальных системах можно использовать текстовые аугментации.

# Текстовые аугментации
Они служат тем же целям, что картиночные, то есть помогают разнообразить выборку и улучшить сходимость модели. Их можно разделить на следующие группы:
* Замена символов — случайные замены/перестановки символов в словах для имитации опечаток.
* Замена слов — случайные перестановки соседних слов/предложений или замена слов на синонимы.
* Обратный перевод — целый текст переводится на какой-то другой язык с помощью сторонней модели, а затем обратно на исходный.